In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models

import xgboost as xgb

In [2]:
# Load dataset
df = pd.read_csv("glucose_dataset_small.csv")

# Features and target
X = df.drop("glucose", axis=1)
y = df["glucose"]

print("Shape:", X.shape)

Shape: (5000, 300)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Save test dataset
test_df = X_test.copy()
test_df["glucose"] = y_test
test_df.to_csv("test.csv", index=False)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4000, 300)
Test shape: (1000, 300)


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save preprocessor
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [5]:
# Reshape for CNN/Transformer (samples, timesteps, channels)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)


def build_hybrid_model(input_shape):
    inputs = layers.Input(shape=input_shape)

    # CNN Block
    x = layers.Conv1D(64, kernel_size=3, activation='relu')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, activation='relu')(x)

    # Transformer Block
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    # Feed Forward
    x_ff = layers.Dense(128, activation='relu')(x)
    x = layers.Add()([x, x_ff])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    # Dense output (feature extractor)
    x = layers.Dense(64, activation='relu')(x)

    model = models.Model(inputs, x)
    return model


feature_extractor = build_hybrid_model((X_train_cnn.shape[1], 1))
feature_extractor.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 298, 64)   │        256 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 149, 64)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 147, 128)  │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 147, 128)  │    131,968 │ conv1d_1[0][0],   │
│ (MultiHeadAttentio… │                   │            │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 147, 128)  │          0 │ conv1d_1[0][0],   │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 147, 128)  │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 147, 128)  │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 147, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 147, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 182,208 (711.75 KB)

 Trainable params: 182,208 (711.75 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Compile feature extractor
feature_extractor.compile(
    optimizer='adam',
    loss='mse'
)

# Train CNN+Transformer
feature_extractor.fit(
    X_train_cnn, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1
)

# Extract features
train_features = feature_extractor.predict(X_train_cnn)
test_features = feature_extractor.predict(X_test_cnn)

# Train XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

xgb_model.fit(train_features, y_train)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - loss: 18524.0723 - val_loss: 15757.7490
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - loss: 14495.5498 - val_loss: 10953.9971
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - loss: 9646.0430 - val_loss: 7275.7148
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - loss: 6447.3794 - val_loss: 5622.2881
Epoch 5/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - loss: 5374.1504 - val_loss: 5204.0767
Epoch 6/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - loss: 5002.1992 - val_loss: 4266.6953
Epoch 7/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 81ms/step - loss: 4130.8018 - val_loss: 4025.9434
Epoch 8/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - loss: 3940.7537 - val_loss: 3984.8247
Epoch 9/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - loss: 3951.8772 - val_loss: 3977.8481
Epoch 10/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - loss: 3974.4753 - val_loss: 3977.5562
Epoch 11/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - loss:

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [7]:
# Predictions
y_pred = xgb_model.predict(test_features)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 30.12066434186829
RMSE: 36.42308320226897
R2 Score: 0.04088033240888245


In [8]:
# Save models
feature_extractor.save("cnn_transformer_model.h5")
joblib.dump(xgb_model, "xgboost_model.pkl")

print("Models saved successfully!")

Models saved successfully!


In [9]:
import joblib
import os

In [10]:
# Load test dataset
test_df = pd.read_csv("test.csv")
os.makedirs("test_records", exist_ok=True)

In [11]:
# Separate features and target
X_test = test_df.drop("glucose", axis=1)
y_test = test_df["glucose"]

# Save each row as a .pkl file
for i in range(len(X_test)):
    record = {
        "features": X_test.iloc[i].values,
        "glucose": y_test.iloc[i]
    }
    
    filename = f"test_records/test_{i}.pkl"
    joblib.dump(record, filename)

print(f"{len(X_test)} test records saved as .pkl files!")

1000 test records saved as .pkl files!
